# Day 68 实战：面试专题（二）—— 项目深挖与 AI 系统设计评估器

> **星辰数科 · 智枢企业级智能体平台 (`v1.0.0-day68`)**

本 Notebook 演示如何使用 `zhishu_agent.system_design_sim` 对企业级 AI 架构方案进行：
1. 高频面试场景加载（100万文档知识库、高并发智能客服、多 Agent 金融研报助手）
2. 弱方案 vs 强方案的六大瓶颈覆盖度与 SLA 评分对比
3. 语义缓存与大小模型路由的月度成本测算
4. 自动生成可直接在面试白板/文档使用的 Mermaid 架构蓝图

In [1]:
import sys
from pathlib import Path

# 自动定位仓库根目录下的 project_iterations/day_68
cur = Path.cwd().resolve()
for parent in [cur, *cur.parents]:
    candidate = parent / 'project_iterations' / 'day_68'
    if candidate.exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from zhishu_agent import (
    ArchitectureProposal,
    estimate_cost,
    evaluate_system_design,
    get_scenario,
    list_scenarios,
)

for s in list_scenarios():
    print(f'[{s.scenario_id}] {s.title} | QPS={s.target_qps} | P99<={s.p99_latency_ms}ms | 预算=¥{s.monthly_budget_cny:,.0f}/月')


[enterprise_kb_1m] 100万文档企业知识库 RAG 平台 | QPS=200 | P99<=1200ms | 预算=¥35,000/月
[high_concurrency_cs] 高并发电商/金融智能客服系统 | QPS=1000 | P99<=800ms | 预算=¥85,000/月
[multi_agent_fin_research] 多 Agent 金融研报分析助手 | QPS=50 | P99<=8000ms | 预算=¥45,000/月


## 1. 查看「100万文档企业知识库」面试官必问追问清单

In [2]:
kb_scenario = get_scenario('enterprise_kb_1m')
print(f'数据规模：{kb_scenario.data_scale}')
print('面试官高频追问：')
for idx, q in enumerate(kb_scenario.key_follow_up_questions, 1):
    print(f'  {idx}. {q}')


数据规模：100万文档（约 2,000 万 Chunks，向量维度 1024）
面试官高频追问：
  1. 2000 万向量下如何做多租户/部门权限过滤，才能不破坏 HNSW 图连通性导致召回暴跌？
  2. 文档每日增量更新 1 万篇，如何保证索引近实时生效且不阻塞在线检索？
  3. Cross-Encoder Rerank 耗时 180ms，如何在 P99 < 1.2s 内平衡精度与延迟？


## 2. 评估对比：初级直连方案 vs 智枢高可用生产方案

In [3]:
weak = ArchitectureProposal(
    proposal_name='初级直连方案',
    scenario_id='enterprise_kb_1m',
    components=['FastAPI', '单机 FAISS', 'GPT-4o API'],
    target_qps=50,
    expected_p99_ms=2600,
)

strong = ArchitectureProposal(
    proposal_name='智枢高可用 RAG 生产方案 v2.0',
    scenario_id='enterprise_kb_1m',
    components=[
        'API Gateway 令牌桶限流',
        'Redis Semantic Cache',
        'Milvus HNSW 分片 + BM25 混合检索 + Cross-Encoder Rerank',
        '大小模型智能 Router',
        'Langfuse + OpenTelemetry + Prometheus 可观测',
    ],
    target_qps=250,
    expected_p99_ms=950,
    caching='Redis 向量语义缓存 (cosine >= 0.92)',
    fallback='Circuit Breaker 熔断降级至本地 7B 模型',
    guardrails='PII 脱敏 + Prompt 注入拦截 + Citation 校验',
    cost_control='大小模型路由 (65% 走 7B) + 租户配额',
)

r_weak = evaluate_system_design(weak)
r_strong = evaluate_system_design(strong)

print(f'弱方案评分: {r_weak.overall_score}/100 ({r_weak.grade}) | 缺失瓶颈: {r_weak.missing_bottlenecks}')
print(f'强方案评分: {r_strong.overall_score}/100 ({r_strong.grade}) | 缺失瓶颈: {r_strong.missing_bottlenecks}')
print(f'月成本对比: ¥{r_weak.cost_estimate.estimated_monthly_cost_cny:,.2f} -> ¥{r_strong.cost_estimate.estimated_monthly_cost_cny:,.2f}')


弱方案评分: 15/100 (D) | 缺失瓶颈: ['semantic_cache', 'rate_limiting', 'circuit_breaker', 'observability', 'cost_routing']
强方案评分: 100/100 (S) | 缺失瓶颈: []
月成本对比: ¥67,500.00 -> ¥18,525.00


## 3. 输出标杆方案的 Mermaid 架构蓝图

In [4]:
print(r_strong.mermaid_blueprint)


flowchart LR
    U["用户请求 (200 QPS / P99<=1200ms)"] --> GW["API Gateway (限流/鉴权)"]
    GW --> GUARD["Guardrails (PII 脱敏 + Prompt 注入拦截 +...)"]
    GUARD --> CACHE["Redis Semantic Cache (命中直出)"]
    CACHE -- "Cache Hit" --> RESP["流式响应 (SSE)"]
    CACHE -- "Cache Miss" --> RET["Hybrid Vector DB (HNSW + BM25 + Rerank)"]
    RET --> RTR["Model Router (大小模型分级)"]
    RTR --> CB["Circuit Breaker & Fallback (熔断降级)"]
    CB --> RESP
    GW -. "Metrics & Trace" .-> OBS["Observability (OTel / Langfuse / Prometheus)"]
    RTR -. "Token & Latency" .-> OBS
